# Dataset 1 - Klasifikasi Penyakit Kardiovaskular
Notebook berisi pipeline ML lengkap untuk dataset Cardiovascular Disease (target biner `cardio`) dengan dua skenario split 80:20 dan 70:30, validasi silang 5-fold, dan hyperparameter tuning RandomizedSearchCV.


## 1. Setup dan Konfigurasi
Memuat library, menetapkan random seed, dan menyiapkan fungsi pembantu untuk menyimpan figure SVG dan tabel CSV.


In [13]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import randint, uniform
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = Path("data/cardio_train.csv")
RESULT_DIR = Path("results/classification")
FIGURE_DIR = RESULT_DIR / "figures"
TABLE_DIR = RESULT_DIR / "tables"
MODEL_DIR = RESULT_DIR / "models"
for directory in [FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"


def save_svg(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, format="svg")
    plt.close(fig)
    return path


def show_table(df, filename=None):
    display(df)
    if filename:
        df.to_csv(TABLE_DIR / filename, index=False)


def save_table(df, filename, index=False):
    df.to_csv(TABLE_DIR / filename, index=index)


## 2. Load Data
Mendukung path lokal maupun Google Colab, jika dijalankan di Colab, dataset dimuat dari Google Drive.


In [14]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = Path("/content/drive/MyDrive/UTS_DEEPL_S2/data/cardio_train.csv")
    RESULT_DIR = Path("/content/drive/MyDrive/UTS_DEEPL_S2/results/classification")
    FIGURE_DIR = RESULT_DIR / "figures"
    TABLE_DIR  = RESULT_DIR / "tables"
    MODEL_DIR  = RESULT_DIR / "models"
    for directory in [FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH}\n"
        "Pastikan file cardio_train.csv tersedia di lokasi tersebut."
    )
print(f"File ditemukan: {DATA_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File ditemukan: /content/drive/MyDrive/UTS_DEEPL_S2/data/cardio_train.csv


Memuat `cardio_train.csv` dengan separator `;` lalu menampilkan dimensi serta tipe kolom untuk verifikasi awal.


In [15]:
raw_df = pd.read_csv(DATA_PATH, sep=";")
print(f"Shape awal: {raw_df.shape}")
display(raw_df.head())
display(raw_df.info())


Shape awal: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


None

## 3. EDA Awal
Memeriksa missing value, distribusi target biner `cardio`, distribusi numerik tiap fitur, serta boxplot untuk identifikasi outlier klinis sebelum cleaning.


In [24]:
missing_table = (
    raw_df.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing_table["missing_percent"] = missing_table["missing_count"] / len(raw_df) * 100
show_table(missing_table, "missing_values_raw.csv")

target_counts = raw_df["cardio"].value_counts().sort_index().rename_axis("cardio").reset_index(name="count")
target_counts["percent"] = target_counts["count"] / target_counts["count"].sum() * 100
show_table(target_counts, "target_distribution.csv")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=raw_df, x="cardio", ax=ax, palette="Set2")
ax.set_title("Distribusi Target Cardio")
ax.set_xlabel("Cardio")
ax.set_ylabel("Jumlah")
save_svg(fig, "target_distribution.svg")

eda_df = raw_df.assign(
    age_years=raw_df["age"] / 365.25,
    bmi=raw_df["weight"] / ((raw_df["height"] / 100) ** 2),
)
numeric_cols = ["age_years", "height", "weight", "bmi", "ap_hi", "ap_lo"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(data=eda_df, x=col, hue="cardio", kde=True, bins=35, ax=ax)
    ax.set_title(f"Distribusi {col}")
fig.tight_layout()
save_svg(fig, "numeric_distributions_raw.svg")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.boxplot(data=eda_df, y=col, ax=ax, color="#9ecae1")
    ax.set_title(f"Outlier {col}")
save_svg(fig, "outlier_boxplots_raw.svg")


,feature,missing_count,missing_percent
0,id,0,0.0
1,age,0,0.0
2,gender,0,0.0
3,height,0,0.0
4,weight,0,0.0
5,ap_hi,0,0.0
6,ap_lo,0,0.0
7,cholesterol,0,0.0
8,gluc,0,0.0
9,smoke,0,0.0


,cardio,count,percent
0,0,35021,50.03
1,1,34979,49.97


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/classification/figures/outlier_boxplots_raw.svg')

## 4. Feature Engineering dan Pembersihan Outlier
Menambahkan fitur turunan klinis (`age_years`, `bmi`, `pulse_pressure`, `map_pressure`, kategori tekanan darah `bp_category` mengikuti aturan AHA) lalu membuang outlier pada tinggi, berat, dan tekanan darah agar dataset realistis secara medis.


In [17]:
def add_clinical_features(df):
    out = df.copy()
    out["age_years"] = out["age"] / 365.25
    out["bmi"] = out["weight"] / ((out["height"] / 100) ** 2)
    out["pulse_pressure"] = out["ap_hi"] - out["ap_lo"]
    out["map_pressure"] = out["ap_lo"] + (out["ap_hi"] - out["ap_lo"]) / 3.0

    def classify_bp(row):
        sbp, dbp = row["ap_hi"], row["ap_lo"]
        if sbp < 120 and dbp < 80:
            return "normal"
        if 120 <= sbp < 130 and dbp < 80:
            return "elevated"
        if 130 <= sbp < 140 or 80 <= dbp < 90:
            return "htn1"
        if 140 <= sbp < 180 or 90 <= dbp < 120:
            return "htn2"
        return "crisis"

    out["bp_category"] = out.apply(classify_bp, axis=1)
    return out


def prepare_cardio_data(df):
    prepared = add_clinical_features(df)
    valid_mask = (
        prepared["height"].between(120, 220)
        & prepared["weight"].between(30, 200)
        & prepared["ap_hi"].between(80, 240)
        & prepared["ap_lo"].between(40, 160)
        & (prepared["ap_hi"] > prepared["ap_lo"])
    )
    cleaned = (
        prepared.loc[valid_mask]
        .drop(columns=["id", "age"])
        .reset_index(drop=True)
    )
    cleaned = pd.get_dummies(cleaned, columns=["bp_category"], drop_first=True, dtype=int)
    return cleaned, int((~valid_mask).sum())


df, removed_rows = prepare_cardio_data(raw_df)
print(f"Shape setelah pembersihan: {df.shape}")
print(f"Jumlah baris outlier yang dihapus: {removed_rows}")
display(df.head())

clean_summary = pd.DataFrame(
    {
        "metric": ["raw_rows", "clean_rows", "removed_rows", "removed_percent"],
        "value": [len(raw_df), len(df), removed_rows, removed_rows / len(raw_df) * 100],
    }
)
show_table(clean_summary, "cleaning_summary.csv")


Shape setelah pembersihan: (68608, 19)
Jumlah baris outlier yang dihapus: 1392


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,bmi,pulse_pressure,map_pressure,bp_category_elevated,bp_category_htn1,bp_category_htn2,bp_category_normal
0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290,21.967120,30,90.000000,0,1,0,0
1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246,34.927679,50,106.666667,0,0,1,0
2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652,23.507805,60,90.000000,0,1,0,0
3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144,28.710479,50,116.666667,0,0,1,0
4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205,23.011177,40,73.333333,0,0,0,1


,metric,value
0,raw_rows,70000.000000
1,clean_rows,68608.000000
2,removed_rows,1392.000000
3,removed_percent,1.988571


## 5. Korelasi dan Feature Selection
Membuat heatmap korelasi Pearson, mencatat korelasi tiap fitur terhadap target, lalu menghapus pasangan fitur dengan korelasi absolut > 0.95 sebagai dasar feature selection.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3, ax=ax)
ax.set_title("Heatmap Korelasi Dataset Kardiovaskular")
save_svg(fig, "correlation_heatmap.svg")

X_all = df.drop(columns=["cardio"])
y = df["cardio"]

target_corr = (
    corr["cardio"]
    .drop("cardio")
    .rename("correlation_to_target")
    .reset_index()
    .rename(columns={"index": "feature"})
)
target_corr["abs_correlation_to_target"] = target_corr["correlation_to_target"].abs()
target_corr = target_corr.sort_values("abs_correlation_to_target", ascending=False)
show_table(target_corr, "target_correlation.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=target_corr, x="feature", y="abs_correlation_to_target", ax=ax, color="#6baed6")
ax.set_title("Korelasi Absolut Fitur terhadap Target Cardio")
ax.set_xlabel("Fitur")
ax.set_ylabel("|Korelasi terhadap cardio|")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
save_svg(fig, "target_correlation.svg")

corr_abs = X_all.corr(numeric_only=True).abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
target_corr_abs = target_corr.set_index("feature")["abs_correlation_to_target"].to_dict()

redundant_features = set()
redundancy_rows = []
for col in upper.columns:
    high_corr_partners = upper.index[upper[col] > 0.95].tolist()
    for partner in high_corr_partners:
        col_target_corr = target_corr_abs.get(col, 0)
        partner_target_corr = target_corr_abs.get(partner, 0)
        drop_feature = col if col_target_corr <= partner_target_corr else partner
        keep_feature = partner if drop_feature == col else col
        redundant_features.add(drop_feature)
        redundancy_rows.append(
            {
                "removed_feature": drop_feature,
                "kept_feature": keep_feature,
                "pairwise_correlation": upper.loc[partner, col],
                "removed_target_abs_corr": target_corr_abs.get(drop_feature, np.nan),
                "kept_target_abs_corr": target_corr_abs.get(keep_feature, np.nan),
                "reason": "pairwise correlation > 0.95 and lower target relevance",
            }
        )

redundant_features = sorted(redundant_features)
X = X_all.drop(columns=redundant_features)

if redundancy_rows:
    feature_selection_table = pd.DataFrame(redundancy_rows)
else:
    feature_selection_table = pd.DataFrame(
        [
            {
                "removed_feature": "none",
                "kept_feature": "all",
                "pairwise_correlation": np.nan,
                "removed_target_abs_corr": np.nan,
                "kept_target_abs_corr": np.nan,
                "reason": "no redundant feature above threshold",
            }
        ]
    )
show_table(feature_selection_table, "feature_selection.csv")
print(f"Jumlah fitur akhir: {X.shape[1]}")
print(list(X.columns))


,feature,correlation_to_target,abs_correlation_to_target
3,ap_hi,0.428034,0.428034
13,map_pressure,0.409593,0.409593
4,ap_lo,0.340450,0.340450
16,bp_category_htn2,0.339342,0.339342
12,pulse_pressure,0.336932,0.336932
10,age_years,0.239354,0.239354
5,cholesterol,0.221430,0.221430
17,bp_category_normal,-0.219733,0.219733
11,bmi,0.189538,0.189538
2,weight,0.180058,0.180058


,removed_feature,kept_feature,pairwise_correlation,removed_target_abs_corr,kept_target_abs_corr,reason
0,none,all,NaN,NaN,NaN,no redundant feature above threshold


Jumlah fitur akhir: 18
['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'age_years', 'bmi', 'pulse_pressure', 'map_pressure', 'bp_category_elevated', 'bp_category_htn1', 'bp_category_htn2', 'bp_category_normal']


## 6. Perbandingan Sebelum dan Sesudah Feature Engineering
Menjalankan validasi silang 5-fold pada Logistic Regression untuk dataset mentah dan dataset hasil feature engineering supaya dampak FE pada CV F1, accuracy, dan ROC-AUC dapat diukur.


In [19]:
compare_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
compare_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)
compare_scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}

raw_X_compare = raw_df.drop(columns=["id", "cardio"])
raw_y_compare = raw_df["cardio"]

feature_engineering_rows = []
for stage, features, target in [
    ("before_feature_engineering", raw_X_compare, raw_y_compare),
    ("after_feature_engineering", X, y),
]:
    cv_result = cross_validate(
        compare_model,
        features,
        target,
        scoring=compare_scoring,
        cv=compare_cv,
        n_jobs=1,
        error_score="raise",
    )
    feature_engineering_rows.append(
        {
            "stage": stage,
            "rows": len(features),
            "features": features.shape[1],
            "cv_accuracy": cv_result["test_accuracy"].mean(),
            "cv_f1": cv_result["test_f1"].mean(),
            "cv_roc_auc": cv_result["test_roc_auc"].mean(),
        }
    )

feature_engineering_comparison = pd.DataFrame(feature_engineering_rows)
show_table(feature_engineering_comparison, "feature_engineering_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=feature_engineering_comparison, x="stage", y="cv_f1", ax=ax, palette="Set2")
ax.set_title("Perbandingan CV F1 Sebelum dan Sesudah Feature Engineering")
ax.set_xlabel("Tahap")
ax.set_ylabel("CV F1 Logistic Regression")
ax.tick_params(axis="x", rotation=15)
save_svg(fig, "feature_engineering_comparison.svg")


,stage,rows,features,cv_accuracy,cv_f1,cv_roc_auc
0,before_feature_engineering,70000,11,0.721071,0.708545,0.784277
1,after_feature_engineering,68608,18,0.726854,0.704874,0.793035


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/classification/figures/feature_engineering_comparison.svg')

## 7. Baseline Lima Model
Melatih Logistic Regression, KNN, Decision Tree, Random Forest, dan Linear SVM dengan StratifiedKFold k=5, lalu mengevaluasi accuracy, precision, recall, F1, dan ROC-AUC pada test set untuk kedua skenario split (80:20 dan 70:30).


In [20]:
def get_classification_models():
    return {
        "Logistic Regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
            ]
        ),
        "KNN": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=21, weights="distance")),
            ]
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            n_estimators=120,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
            n_jobs=1,
        ),
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LinearSVC(C=1.0, class_weight="balanced", random_state=RANDOM_STATE, max_iter=8000, dual="auto")),
            ]
        ),
    }


def model_scores(model, X_test, y_test):
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        y_score = y_pred

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
    }


splits = {"80:20": 0.20, "70:30": 0.30}
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

baseline_rows = []
split_data = {}

for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y,
    )
    split_data[split_name] = (X_train, X_test, y_train, y_test)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for model_name, model in get_classification_models().items():
        cv_result = cross_validate(
            model, X_train, y_train, scoring=scoring, cv=cv, n_jobs=1, error_score="raise",
        )
        fitted_model = clone(model).fit(X_train, y_train)
        test_metrics = model_scores(fitted_model, X_test, y_test)
        baseline_rows.append(
            {
                "split": split_name,
                "model": model_name,
                "cv_accuracy": cv_result["test_accuracy"].mean(),
                "cv_precision": cv_result["test_precision"].mean(),
                "cv_recall": cv_result["test_recall"].mean(),
                "cv_f1": cv_result["test_f1"].mean(),
                "cv_roc_auc": cv_result["test_roc_auc"].mean(),
                "test_accuracy": test_metrics["accuracy"],
                "test_precision": test_metrics["precision"],
                "test_recall": test_metrics["recall"],
                "test_f1": test_metrics["f1"],
                "test_roc_auc": test_metrics["roc_auc"],
            }
        )

baseline_results = pd.DataFrame(baseline_rows).sort_values(["split", "cv_f1"], ascending=[True, False])
show_table(baseline_results, "baseline_results.csv")


,split,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
5,70:30,Logistic Regression,0.726830,0.757064,0.659497,0.704890,0.792930,0.726668,0.757080,0.658941,0.704610,0.792647
6,70:30,KNN,0.716627,0.728679,0.680668,0.703847,0.775819,0.720109,0.731956,0.685162,0.707786,0.779828
8,70:30,Random Forest,0.709235,0.712847,0.690475,0.701457,0.769029,0.710149,0.711803,0.695866,0.703744,0.771436
9,70:30,SVM,0.725476,0.761803,0.647628,0.700066,0.792407,0.725356,0.761246,0.648139,0.700154,0.791998
7,70:30,Decision Tree,0.630796,0.626204,0.629404,0.627794,0.630795,0.639071,0.633353,0.642247,0.637769,0.639135
1,80:20,KNN,0.718544,0.730831,0.682393,0.705770,0.775793,0.717680,0.731312,0.678745,0.704049,0.779382
0,80:20,Logistic Regression,0.726488,0.756932,0.658638,0.704355,0.792628,0.726643,0.760192,0.653705,0.702938,0.793990
3,80:20,Random Forest,0.709124,0.712641,0.690495,0.701372,0.769305,0.710756,0.715399,0.689792,0.702362,0.772541
4,80:20,SVM,0.726360,0.763052,0.648142,0.700908,0.792083,0.724603,0.763757,0.641921,0.697559,0.793336
2,80:20,Decision Tree,0.630288,0.626292,0.626598,0.626442,0.630224,0.628771,0.626474,0.618353,0.622387,0.628699


## 8. Visualisasi Baseline
Membandingkan CV F1 dan test ROC-AUC antar model dalam dua split untuk mengidentifikasi kandidat terbaik yang akan dituning lebih lanjut.


In [21]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=baseline_results, x="model", y="cv_f1", hue="split", ax=ax, palette="Set2")
ax.set_title("Perbandingan CV F1 Baseline")
ax.set_xlabel("Model")
ax.set_ylabel("CV F1")
ax.tick_params(axis="x", rotation=20)
save_svg(fig, "baseline_cv_f1_comparison.svg")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=baseline_results, x="model", y="test_roc_auc", hue="split", ax=ax, palette="Set3")
ax.set_title("Perbandingan Test ROC-AUC Baseline")
ax.set_xlabel("Model")
ax.set_ylabel("Test ROC-AUC")
ax.tick_params(axis="x", rotation=20)
save_svg(fig, "baseline_test_roc_auc_comparison.svg")


PosixPath('/content/drive/MyDrive/UTS_DEEPL_S2/results/classification/figures/baseline_test_roc_auc_comparison.svg')

## 9. Hyperparameter Tuning
Menjalankan RandomizedSearchCV (n_iter=60, scoring=F1) pada dua kandidat: Logistic Regression (model linear terbaik) dan Random Forest (kandidat non-linear), untuk kedua split. Best params dan best CV F1 kedua kandidat dicatat agar perbandingan adil.


In [22]:
overall_model_score = (
    baseline_results.groupby("model", as_index=False)["cv_f1"]
    .mean()
    .sort_values("cv_f1", ascending=False)
)
print("Ranking model berdasarkan rata-rata CV F1 dua split:")
display(overall_model_score)
overall_model_score.to_csv(TABLE_DIR / "overall_model_score.csv", index=False)

tuning_candidates = ["Logistic Regression", "Random Forest"]


def tuning_space(model_name):
    if model_name == "Logistic Regression":
        estimator = get_classification_models()[model_name]
        params = {
            "model__C": uniform(0.01, 10.0),
            "model__solver": ["lbfgs", "liblinear"],
            "model__class_weight": ["balanced", None],
        }
    elif model_name == "Random Forest":
        estimator = get_classification_models()[model_name]
        params = {
            "n_estimators": randint(120, 401),
            "max_depth": [6, 8, 10, 12, 14, None],
            "min_samples_split": randint(2, 21),
            "min_samples_leaf": randint(1, 11),
            "max_features": ["sqrt", "log2", None],
        }
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return estimator, params


N_ITER = 60
tuning_rows = []
best_estimators = {}

for split_name, (X_train, X_test, y_train, y_test) in split_data.items():
    candidate_results = []
    for cand_name in tuning_candidates:
        estimator, params = tuning_space(cand_name)
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=params,
            n_iter=N_ITER,
            scoring="f1",
            cv=5,
            random_state=RANDOM_STATE,
            n_jobs=1,
            refit=True,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        tuned_model = search.best_estimator_
        tuned_metrics = model_scores(tuned_model, X_test, y_test)
        baseline_row = baseline_results[
            (baseline_results["split"] == split_name) & (baseline_results["model"] == cand_name)
        ].iloc[0]
        candidate_results.append({
            "split": split_name,
            "model": cand_name,
            "baseline_cv_f1": baseline_row["cv_f1"],
            "baseline_test_f1": baseline_row["test_f1"],
            "tuned_best_cv_f1": search.best_score_,
            "tuned_test_f1": tuned_metrics["f1"],
            "tuned_test_accuracy": tuned_metrics["accuracy"],
            "tuned_test_precision": tuned_metrics["precision"],
            "tuned_test_recall": tuned_metrics["recall"],
            "tuned_test_roc_auc": tuned_metrics["roc_auc"],
            "best_params": json.dumps({k: (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()}, default=str),
            "fitted_estimator": tuned_model,
        })
    candidate_results.sort(key=lambda r: r["tuned_test_f1"], reverse=True)
    winner = candidate_results[0]
    best_estimators[split_name] = winner["fitted_estimator"]
    for r in candidate_results:
        row = {k: v for k, v in r.items() if k != "fitted_estimator"}
        row["selected_for_split"] = (r["model"] == winner["model"])
        tuning_rows.append(row)

tuning_results = pd.DataFrame(tuning_rows)
show_table(tuning_results, "tuning_results.csv")
joblib.dump(best_estimators, MODEL_DIR / "best_classification_estimators.joblib")
print(f"\nN_iter RandomizedSearchCV: {N_ITER}")
print("Pemenang per split (berdasar test F1):")
for split_name, est in best_estimators.items():
    print(f"  {split_name}: {type(est).__name__}")


Ranking model berdasarkan rata-rata CV F1 dua split:


,model,cv_f1
1,KNN,0.704809
2,Logistic Regression,0.704623
3,Random Forest,0.701414
4,SVM,0.700487
0,Decision Tree,0.627118


,split,model,baseline_cv_f1,baseline_test_f1,tuned_best_cv_f1,tuned_test_f1,tuned_test_accuracy,tuned_test_precision,tuned_test_recall,tuned_test_roc_auc,best_params,selected_for_split
0,80:20,Random Forest,0.701372,0.702362,0.720093,0.720320,0.735097,0.754027,0.689498,0.801164,"{""max_depth"": 10, ""max_features"": null, ""min_s...",True
1,80:20,Logistic Regression,0.704355,0.702938,0.704409,0.702938,0.726643,0.760192,0.653705,0.793989,"{""model__C"": 0.7555064367977082, ""model__class...",False
2,70:30,Random Forest,0.701457,0.703744,0.720335,0.724305,0.736627,0.751160,0.699303,0.801382,"{""max_depth"": 8, ""max_features"": null, ""min_sa...",True
3,70:30,Logistic Regression,0.704890,0.704610,0.704180,0.704647,0.726716,0.757165,0.658941,0.792643,"{""model__C"": 2.0067378215835974, ""model__class...",False



N_iter RandomizedSearchCV: 60
Pemenang per split (berdasar test F1):
  80:20: RandomForestClassifier
  70:30: RandomForestClassifier


## 10. Confusion Matrix, Classification Report, dan Perbandingan Tuning
Menyimpan confusion matrix dan classification report kedua split sebagai CSV, plot perbandingan F1 sebelum dan sesudah tuning, serta visualisasi confusion matrix dalam SVG.


In [23]:
selected_rows = tuning_results[tuning_results["selected_for_split"] == True]
comparison_rows = []
for _, row in selected_rows.iterrows():
    comparison_rows.extend(
        [
            {"split": row["split"], "stage": "baseline", "model": row["model"], "f1": row["baseline_test_f1"]},
            {"split": row["split"], "stage": "tuned",    "model": row["model"], "f1": row["tuned_test_f1"]},
        ]
    )
comparison_df = pd.DataFrame(comparison_rows)
save_table(comparison_df, "tuning_f1_comparison.csv")

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=comparison_df, x="split", y="f1", hue="stage", ax=ax, palette="Set2")
ax.set_title("Perbandingan Test F1 Sebelum dan Sesudah Tuning")
ax.set_xlabel("Split")
ax.set_ylabel("Test F1")
save_svg(fig, "tuning_f1_comparison.svg")

for split_name, tuned_model in best_estimators.items():
    _, X_test, _, y_test = split_data[split_name]
    y_pred = tuned_model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])
    cm_df.to_csv(TABLE_DIR / f"confusion_matrix_tuned_{split_name.replace(':', '_')}.csv")

    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).T
    report_df.to_csv(TABLE_DIR / f"classification_report_tuned_{split_name.replace(':', '_')}.csv")

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix Tuned - Split {split_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    save_svg(fig, f"confusion_matrix_tuned_{split_name.replace(':', '_')}.svg")
